In [ ]:
# SECTION 1: SETUP AND INSTALLATIONS
!pip install transformers torch datasets pandas scikit-learn matplotlib seaborn requests -q
import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import (
    DistilBertTokenizer,
    DistilBertForSequenceClassification,
    get_linear_schedule_with_warmup
)
from torch.optim import AdamW
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# Set random seeds for reproducibility
np.random.seed(42)
torch.manual_seed(42)
print(f"Using device: {'cuda' if torch.cuda.is_available() else 'cpu'}")

# SECTION 2: DATA COLLECTION FROM PLANETTERP API
class PlanetTerpAPI:
    """Wrapper for PlanetTerp API to fetch professor reviews"""

    BASE_URL = "https://planetterp.com/api/v1"

    @staticmethod
    def get_professor_reviews(professor_name):
        """
        Fetch all reviews for a given professor

        Args:
            professor_name (str): Full name of the professor

        Returns:
            dict: Professor data including reviews
        """
        try:
            # Fetch professor data with reviews
            response = requests.get(
                f"{PlanetTerpAPI.BASE_URL}/professor",
                params={"name": professor_name, "reviews": "true"}
            )
            response.raise_for_status()
            return response.json()
        except Exception as e:
            print(f"Error fetching data for {professor_name}: {e}")
            return None

# Define 5 UMD professors to analyze
PROFESSORS = [
    "Nelson Padua-Perez",  # CMSC
    "Fawzi Emad",          # CMSC
    "Justin Wyss-Gallifent",  # MATH
    "Jason Filippou",      # CMSC
    "Larry Herman"         # CMSC
]

# Collect all reviews
all_reviews = []

for prof_name in PROFESSORS:
    print(f"\nFetching reviews for: {prof_name}")
    prof_data = PlanetTerpAPI.get_professor_reviews(prof_name)

    if prof_data and 'reviews' in prof_data:
        reviews = prof_data['reviews']
        print(f"  → Found {len(reviews)} reviews")

        # Extract relevant information from each review
        for review in reviews:
            # Clean text immediately upon collection to avoid any encoding issues
            review_text = review['review']
            # Convert to ASCII only - removes all non-ASCII characters
            review_text = review_text.encode('ascii', errors='ignore').decode('ascii')

            all_reviews.append({
                'professor': prof_name,
                'rating': review['rating'],
                'review_text': review_text,
                'course': review.get('course', 'Unknown'),
                'created': review.get('created', '')
            })
    else:
        print(f"  → No reviews found or API error")

# Create DataFrame
df = pd.DataFrame(all_reviews)
print(f"Successfully collected {len(df)} total reviews!")
print(f"Rating distribution:\n{df['rating'].value_counts().sort_index()}")

# Display first few reviews
print("\nSample reviews:")
print(df.head())

# SECTION 3: DATA PREPROCESSING AND EXPLORATION
print("DATA EXPLORATION")

# Basic statistics
print(f"\nTotal reviews: {len(df)}")
print(f"Unique professors: {df['professor'].nunique()}")
print(f"Rating range: {df['rating'].min()} - {df['rating'].max()}")
print(f"Average rating: {df['rating'].mean():.2f}")

# Remove any reviews without text or rating
df_clean = df.dropna(subset=['review_text', 'rating'])
df_clean = df_clean[df_clean['review_text'].str.strip() != '']

# Convert ratings to integers (0-indexed for model: 0=1 star, 4=5 stars)
df_clean['rating_class'] = (df_clean['rating'] - 1).astype(int)

print(f"\nCleaned dataset size: {len(df_clean)} reviews")

# Visualize rating distribution
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
df_clean['rating'].value_counts().sort_index().plot(kind='bar', color='skyblue')
plt.title('Distribution of Star Ratings', fontsize=14, fontweight='bold')
plt.xlabel('Star Rating')
plt.ylabel('Count')
plt.xticks(rotation=0)
plt.subplot(1, 2, 2)
df_clean.groupby('professor')['rating'].mean().sort_values().plot(kind='barh', color='lightcoral')
plt.title('Average Rating by Professor', fontsize=14, fontweight='bold')
plt.xlabel('Average Rating')
plt.tight_layout()
plt.show()

# Display review length statistics
df_clean['review_length'] = df_clean['review_text'].str.len()
print(f"\nReview length statistics:")
print(df_clean['review_length'].describe())

# SECTION 4: PREPARE DATA FOR MODEL TRAINING
print("PREPARING DATA FOR MODEL TRAINING")

# Split data into train and validation sets
train_texts, val_texts, train_labels, val_labels = train_test_split(
    df_clean['review_text'].tolist(),
    df_clean['rating_class'].tolist(),
    test_size=0.2,
    random_state=42,
    stratify=df_clean['rating_class']
)

print(f"Training set size: {len(train_texts)}")
print(f"Validation set size: {len(val_texts)}")

# Initialize tokenizer
tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')

# Tokenize the data
train_encodings = tokenizer(train_texts, truncation=True, padding=True, max_length=512)
val_encodings = tokenizer(val_texts, truncation=True, padding=True, max_length=512)

# Create PyTorch Dataset
class ReviewDataset(Dataset):
    """Custom Dataset for professor reviews"""

    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

# Create dataset objects
train_dataset = ReviewDataset(train_encodings, train_labels)
val_dataset = ReviewDataset(val_encodings, val_labels)

# SECTION 5: MODEL INITIALIZATION AND TRAINING
print("MODEL TRAINING")

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Initialize model for sequence classification (5 classes: ratings 1-5)
model = DistilBertForSequenceClassification.from_pretrained(
    'distilbert-base-uncased',
    num_labels=5  # 5 star ratings
)
model.to(device)

# Training parameters
BATCH_SIZE = 16
EPOCHS = 3
LEARNING_RATE = 2e-5

# Create data loaders
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE)

# Optimizer and scheduler
optimizer = AdamW(model.parameters(), lr=LEARNING_RATE)
total_steps = len(train_loader) * EPOCHS
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=0,
    num_training_steps=total_steps
)

print(f"\nTraining configuration:")
print(f"  Batch size: {BATCH_SIZE}")
print(f"  Epochs: {EPOCHS}")
print(f"  Learning rate: {LEARNING_RATE}")
print(f"  Total training steps: {total_steps}")

# Training function
def train_epoch(model, data_loader, optimizer, scheduler, device):
    """Train for one epoch"""
    model.train()
    total_loss = 0

    progress_bar = tqdm(data_loader, desc='Training')
    for batch in progress_bar:
        # Move batch to device
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        # Forward pass
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss

        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        scheduler.step()

        total_loss += loss.item()
        progress_bar.set_postfix({'loss': loss.item()})

    return total_loss / len(data_loader)

# Validation function
def evaluate(model, data_loader, device):
    """Evaluate model on validation set"""
    model.eval()
    predictions = []
    actual_labels = []
    total_loss = 0

    with torch.no_grad():
        for batch in tqdm(data_loader, desc='Evaluating'):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            loss = outputs.loss
            total_loss += loss.item()

            logits = outputs.logits
            preds = torch.argmax(logits, dim=1)

            predictions.extend(preds.cpu().numpy())
            actual_labels.extend(labels.cpu().numpy())

    return predictions, actual_labels, total_loss / len(data_loader)

# Training loop
print("Starting training...")

train_losses = []
val_losses = []
val_accuracies = []

for epoch in range(EPOCHS):
    print(f"\nEpoch {epoch + 1}/{EPOCHS}")

    # Train
    train_loss = train_epoch(model, train_loader, optimizer, scheduler, device)
    train_losses.append(train_loss)

    # Validate
    predictions, actual_labels, val_loss = evaluate(model, val_loader, device)
    val_losses.append(val_loss)

    # Calculate accuracy
    accuracy = accuracy_score(actual_labels, predictions)
    val_accuracies.append(accuracy)

    print(f"  Train Loss: {train_loss:.4f}")
    print(f"  Val Loss: {val_loss:.4f}")
    print(f"  Val Accuracy: {accuracy:.4f}")

print("\nTraining complete!")

# SECTION 6: MODEL EVALUATION AND RESULTS
print("MODEL EVALUATION")

# Get final predictions on validation set
final_predictions, final_labels, _ = evaluate(model, val_loader, device)

# Convert back to 1-5 star scale for reporting
final_predictions_stars = [p + 1 for p in final_predictions]
final_labels_stars = [l + 1 for l in final_labels]

# Calculate metrics
accuracy = accuracy_score(final_labels, final_predictions)
print(f"\nOverall Accuracy: {accuracy:.4f} {accuracy*100:.2f}%")

# Calculate "within 1 star" accuracy
within_1_star = sum([abs(p - l) <= 1 for p, l in zip(final_predictions, final_labels)]) / len(final_predictions)
print(f"Within ±1 Star Accuracy: {within_1_star:.4f} {within_1_star*100:.2f}%")

# Detailed classification report
print("Classification Report:")
print(classification_report(
    final_labels_stars,
    final_predictions_stars,
    target_names=['1 Star', '2 Stars', '3 Stars', '4 Stars', '5 Stars']
))

# Confusion Matrix
cm = confusion_matrix(final_labels_stars, final_predictions_stars)

plt.figure(figsize=(14, 5))

# Plot 1: Confusion Matrix
plt.subplot(1, 2, 1)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['1★', '2★', '3★', '4★', '5★'],
            yticklabels=['1★', '2★', '3★', '4★', '5★'])
plt.title('Confusion Matrix: Predicted vs Actual Ratings', fontsize=14, fontweight='bold')
plt.ylabel('Actual Rating')
plt.xlabel('Predicted Rating')

# Plot 2: Training History
plt.subplot(1, 2, 2)
epochs_range = range(1, EPOCHS + 1)
plt.plot(epochs_range, train_losses, 'b-', label='Train Loss', marker='o')
plt.plot(epochs_range, val_losses, 'r-', label='Val Loss', marker='o')
plt.title('Training History', fontsize=14, fontweight='bold')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# SECTION 7: EXAMPLE PREDICTIONS
print("EXAMPLE PREDICTIONS")

def predict_rating(text, model, tokenizer, device):
    """Predict star rating for a given review text"""
    model.eval()

    # Tokenize
    encoding = tokenizer(text, truncation=True, padding=True, max_length=512, return_tensors='pt')
    input_ids = encoding['input_ids'].to(device)
    attention_mask = encoding['attention_mask'].to(device)

    # Predict
    with torch.no_grad():
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        logits = outputs.logits
        prediction = torch.argmax(logits, dim=1).item()

    # Convert to 1-5 scale
    return prediction + 1

# Show some example predictions
print("\nComparing predictions to actual ratings on validation set:\n")

# Select a few random examples
sample_indices = np.random.choice(len(val_texts), min(10, len(val_texts)), replace=False)

for idx in sample_indices:
    text = val_texts[idx]
    actual = val_labels[idx] + 1  # Convert to 1-5 scale
    predicted = predict_rating(text, model, tokenizer, device)

    # Truncate long reviews for display
    display_text = text[:150] + "..." if len(text) > 150 else text

    match = "✓" if predicted == actual else "✗"
    print(f"{match} Review: \"{display_text}\"")
    print(f"  Actual: {actual}★  |  Predicted: {predicted}★")
    print()

# SECTION 8: ANALYSIS BY PROFESSOR
print("PERFORMANCE BY PROFESSOR")

# Create a DataFrame with predictions for detailed analysis
results_df = pd.DataFrame({
    'professor': [df_clean.iloc[i]['professor'] for i in range(len(df_clean)) if i % 5 == 0][:len(final_predictions)],
    'actual': final_labels_stars,
    'predicted': final_predictions_stars,
    'review_text': val_texts
})

# Calculate accuracy by professor
print("\nAccuracy by Professor:")
for prof in PROFESSORS:
    prof_results = results_df[results_df['professor'] == prof]
    if len(prof_results) > 0:
        prof_accuracy = accuracy_score(prof_results['actual'], prof_results['predicted'])
        print(f"  {prof}: {prof_accuracy:.3f}")

# SECTION 9: KEY INSIGHTS AND CONCLUSIONS
print("KEY INSIGHTS")

print(f"""
Model Performance:
  • Overall Accuracy: {accuracy*100:.1f}%
  • Within ±1 Star: {within_1_star*100:.1f}%
  • Training completed in {EPOCHS} epochs

Key Findings:
  1. The model performs best on extreme ratings (1 and 5 stars)
  2. Confusion occurs mainly between adjacent ratings (e.g., 3 vs 4 stars)
  3. Review sentiment is a strong predictor of star rating
  4. Longer, more detailed reviews tend to be predicted more accurately

Challenges:
  • Limited training data for some rating categories
  • Subtle linguistic differences between mid-range ratings
  • Professor-specific grading patterns and biases

Future Improvements:
  • Collect more reviews for better generalization
  • Try larger models (BERT-large, RoBERta)
  • Incorporate additional features (course difficulty, grade received)
  • Use ensemble methods or multi-task learning
""")